# Assignment 7: RAG with Ollama

In this assignment, we explore **Retrieval-augmented generation (RAG)** with Large Language Models (LLMs). **This assignment is completely manually graded.**

Before you start, please be adivsed that you will need [Ollama](https://ollama.com/) and quite a few Python packages. It is therefore recommended that you use the [JupyterHub at Minerva cluster](https://git.chalmers.se/karppa/minerva/) as **Ollama is already running on the server and all required packges are installed**. See instructions behind the link to see how to gain access.

The next section describes how to install the prerequisites on your local computer. You can skip it if you are using Minerva.

## Prerequisites (only for local installation)

**You can skip this section if you use Minerva.**

To run this assignment locally, you are going to need to install quite a few prerequisites. Most of these can be installed on Conda throught the conda-forge channel (`conda install -c conda-forge` *package name*). It is recommended that you create a new environment for this purpose. Install the following packages through conda forge:
- `langchain`
- `langchain-community`
- `langchain-huggingface`
- `langchain-chroma`
- `langchain-ollama`
- `pypdf`
- `chromadb`

The package `langchain-ollama` is also needed but is not available through Conda. Install it with `pip`:

```python3 -m pip install langchain-ollama```

Finally, you will need to install [Ollama](https://ollama.com/). It is also available through Conda, but the version served there is rather dated and does not support newer LLMs (it works fine with [Mistral](https://mistral.ai/) 7B, though).

Ollama is open source server software that runs LLMs locally on your computer. You can choose which LLM you want to use. By default, the code below uses [Llama](https://www.llama.com/) 3.2 3B, but if this is slow, you can switch to Llama 3.2 1B, or if you want to use an European model (or want to use an older version of Ollama), you can try Mistral 7B.

## Introduction

Below, you can see a class that has been named `RAGAssistant`. It provides a convenient access to try out RAG. It works as follows:
- The constructor simply initializes the class (nothing fancy).
- `add_document(filename)` adds the given document file into the database. The document must either be raw UTF-8 text (`.txt`) or a PDF file (`.pdf`). What it does is that it extracts the text from the document, converts it into chunks, embeds them into an LLM vector space, and then stores them in a [Chroma](https://www.trychroma.com/) database.
- `query(query_text)` queries the database using the specified LLM. What it does is that it first queries the closest documents matching the query from the database, injects these as a context into the prompt presented to the LLM, and then presents the query to the LLM.
- `reset()` resets the database.

The class can be used in an interactive fashion, so once you've added documents, you can start making as many queries you want. When you want to remove the documents from the database, just use `reset`, and add some new documents instead.

The choice of the LLM is controlled by the variable `LLM`.
**If you are running Ollama locally**, you must `pull` the LLM by issuing the following command on the command line:

```ollama pull llm-name```

E.g., `ollama pull llama3.2:3b` pulls (dowloads from the Internet) the default LLM we might want to use.

The following models have been installed on Minerva:
- `mistral:7b`
- `llama3.1:8b`
- `llama3.2:1b`
- `llama3.2:3b`

The system is very simple and does not, for example, contain any memory to process multiple queries. However, it is very easy to extend this to more complex applications.

**Beware:** LLMs are very computationally demanding, and the queries may be slow. Furthermore, in some cases, the queries can take an extraordinary amount of time in which case it might be wise to interrupt the query and rephrase. `llama3.2:1b` is probably the fastest of the bunch, which of course means that it performs the poorest, but any one will do for our purposes.

**Beware:** Adding documents into the database can take a lot of time because of the expensive embedding computations. When testing the Question 3 on Minerva, it took **over 7 minutes** to add the entire dataset (all PDF files). **This is normal.** However, you only need to add the documents once during a session and then you can query as many prompts as you wish.

In [26]:
import shutil
import os
from langchain_community.document_loaders.pdf import PyPDFLoader, PyPDFDirectoryLoader
from langchain_community.document_loaders.text import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

class RAGAssistant:
    """
    Performs queries about documents using Mistral 7B model with Ollama
    """

    CHROMA_PATH = 'chroma'
    EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
    PROMPT_TEMPLATE = """
    Answer the question based only on the following context:

    {context}

    ---

    Answer the question based on the above context: {question}
    """
    # LLM = 'mistral:7b'
    # LLM = 'llama3.1:8b'
    # LLM = 'llama3.2:1b'
    LLM = 'llama3.2:3b'

    def __init__(self):
        """
        Initialization: simply reset the database (remove it)
        """
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=800,
            chunk_overlap=80,
            length_function=len,
            is_separator_regex=False,
        )
        self.reset()


    def reset(self):
        """
        Empties the database
        """
        self.db = Chroma(embedding_function = self.get_embedding_function())
        self.db.reset_collection()

    
    def add_document(self, filename):
        """
        Adds the given document into the database. The file must be either a PDF or a .txt file.
        If filename has no extension, it is assumed to be a directory and all PDF files are loaded
        from the directory.
        """
        if filename.endswith('.pdf'):
            documents = PyPDFLoader(filename).load()
        elif filename.endswith('.txt'):
            documents = TextLoader(filename).load()
        elif os.path.isdir(filename):
            documents = PyPDFDirectoryLoader(filename).load()
        else:
            raise ValueError('Filename must end in either .pdf or .txt, '
                             'or be a directory, '
                                 f'but got `{filename}\'')
            
        chunks = self.text_splitter.split_documents(documents)
        
        chunks_with_ids = self.calculate_chunk_ids(chunks)

        existing_items = self.db.get(include=[])
        existing_ids = set(existing_items["ids"])

        # Only add documents that don't exist in the DB.
        new_chunks = []
        for chunk in chunks_with_ids:
            if chunk.metadata["id"] not in existing_ids:
                new_chunks.append(chunk)

        if len(new_chunks):
            new_chunk_ids = [chunk.metadata["id"] for chunk in new_chunks]
            self.db.add_documents(new_chunks, ids=new_chunk_ids)


    def calculate_chunk_ids(self, chunks):
        """
        Helper function for computing chunk ids
        """
        last_page_id = None
        current_chunk_index = 0

        for chunk in chunks:
            source = chunk.metadata.get("source")
            page = chunk.metadata.get("page")
            current_page_id = f"{source}:{page}"
            if current_page_id == last_page_id:
                current_chunk_index += 1
            else:
                current_chunk_index = 0
            chunk_id = f"{current_page_id}:{current_chunk_index}"
            last_page_id = current_page_id
            chunk.metadata["id"] = chunk_id

        return chunks

    def get_embedding_function(self):
        """
        Helper function for simply returning the same embedding function always
        """
        return HuggingFaceEmbeddings(model_name=self.EMBEDDING_MODEL_NAME)

    def query(self, query_text):
        """
        Performs the RAG query.
        
        First, queries the database for the most relevant documents.
        
        Then, injects the documents into the prompt.
        
        Finally, invokes the LLM to answer the prompt within the context.
        
        Return value: (response,sources) where response is a string and sources is a list of sources
        """
        
        results = self.db.similarity_search_with_score(query_text, k=5)
        context_text = "\n\n---\n\n".join([doc.page_content for doc, _score in results])
        prompt_template = ChatPromptTemplate.from_template(self.PROMPT_TEMPLATE)
        prompt = prompt_template.format(context=context_text,
                                            question=query_text)
        
        model = ChatOllama(model = self.LLM)
        response_text = model.invoke(prompt)

        sources = [doc.metadata.get("id", None) for doc, _score in results]
        return (response_text.content,sources)


# this is the object we're going to manipulate
rag = RAGAssistant()

## Question 1: EU AI Act

Thhe file `ai_act_en.pdf` (available on Canvas) contains the text of the EU AI Act in English. Use your `rag` object to add the file into the database. Then, choose a question you think is interesting regarding the AI act and explore what the model says about it.

Potential themes can be the aims of the AI act, the impact that the act has on businesses, the impact on particular industries (such as healthcare or finance), or potential issues with the act, or something completely different but relevant. Don't know about the AI act? Now it's time to find out.

Check what documents the model has used for sources. Do the cited sources support the claims the model makes? Once you've probed your model about the act, cross-reference the models answers with other sources you can find online. Do they confirm what the model is saying or are they contradictory, or are the model's answers even nonsensical?

Write 2–4 paragraphs. Include in your answer at least one prompt and answer that your model gave you. Cite the sources you used to confirm or contradict what the model claims.

### Answer 1

We asked for what are the aims of the AI act. The model's answer supported the main ideas mentioned in the EU AI act, which ideally aims to regulate the AI according to the various risk levels. This ensures the high-risk AI systems to follow strict standards for accountability, fairness and transparency. We also see this act emphasis on the importance of public awareness and AI literacy.
 
As a second query we asked how does the AI impact businesses. Although it does not specifically address how the AI Act affects businesses, the text provided mainly describes the rules, specifications, and goals pertaining to AI within a particular legal framework. However, regulation sandboxes, as outlined in the "Blue Guide" press release, can potentially offer companies a methodical framework where they can grow without compromising compliance with law.

You can see our queries, responses to the queries and the sources that the model used to generate responses below.

In [27]:
# you can use as many code cells here as you want, this is manually graded

rag.reset()
rag.add_document("ai_act_en.pdf")

In [28]:
response_eu_act, sources_eu_act = rag.query("What are the aims of the AI act?")

print("Response:")
print(response_eu_act)
print()
print("Sources:")
print(sources_eu_act)

Response:
The text does not explicitly state the aims of the AI Act. However, it can be inferred that the main objectives include:

1. Assessing and regulating AI systems to minimize harm, particularly in high-risk areas such as education and vocational training.
2. Promoting AI literacy among providers, deployers, and affected persons to ensure responsible use of AI systems.
3. Encouraging innovation and development of AI systems while ensuring accountability and transparency.
4. Supporting the Commission in promoting public awareness and understanding of AI benefits and risks.

These objectives are implied from the context and can be inferred from various sections of the text, such as:

* The Commission's role in regulating AI systems (paragraph 1)
* The importance of assessing and mitigating harm caused by AI systems (point b under paragraph 1)
* Promoting AI literacy and public awareness (points f and g under paragraph 1, point v, and section (f))
* Encouraging innovation and devel

In [29]:
response_eu_act_2, sources_eu_act_2 = rag.query("How does the AI act impact businesses?")

print("Response:")
print(response_eu_act_2)
print()
print("Sources:")
print(sources_eu_act_2)

Response:
The provided text does not directly answer how the AI acts impact businesses. It primarily discusses the regulations, requirements, and objectives related to Artificial Intelligence (AI) within a specific legal framework. However, it can be inferred that the implementation of these regulations might have an indirect impact on businesses, particularly in terms of compliance with the rules and ensuring that their innovative AI systems meet the required standards.

In the context of the "Blue Guide" notice mentioned, establishing regulatory sandboxes for AI development and testing could potentially provide a controlled environment for businesses to innovate while meeting regulatory requirements. This might lead to enhanced legal certainty for innovators and, indirectly, support businesses in developing compliant AI solutions.

However, without explicit information on how the AI acts directly impact businesses, it is challenging to provide a more detailed answer. The text focuses

## Question 2: The Coffee Brewer

You have bought a new coffee brewer, but, unfortunately, they only sent the operating instructions in Finnish. The instructions are available in `moccamaster_manual_fi.pdf` (available on Canvas).

Add the document into `rag` and probe it to find out how to operate your coffee machine. Ask questions about things such as safety, operating, and cleaning of the machine, or what one should do if the machine breaks. In light of your understanding of coffee brewers and home appliances in general, do the answers make sense? Discuss and include at least one prompt and one answer from the LLM as part of your answer. Write 2–4 paragraphs.

### Answer 2

We asked for the safety instructions. The model said that there were no specific instructions for safety but it found the important points related to safety. It translated Finnish parts that are included in the answer to English. It seems like the model can understand the document even though it's in Finnish, and can translate the information regarding the query to English.

In the second query, we asked for step by step guidance to prepare the coffee but it couldn't give a good answer. In the first part of its answer, it just mentioned about the features of the coffee machine. It also gave cleaning instructions that were not related to preparing coffee. It also gave this information in Finnish even though the question was in English.

We forgot to add the manual PDF before our first query for this question. There was still AI act document in the database. When we asked about the coffee machine it said there was no information about the coffee in the document. This shows that it can interpret the general context of the document. It's not only trying to find relevant information in the text, it also can understand it's not related to the question.

You can see our queries, responses to the queries and the sources that the model used to generate responses below.

In [31]:
rag.reset()
rag.add_document("moccamaster_manual_fi.pdf")

In [32]:
response_coffee, sources_coffee = rag.query("What are the safety instructions to use the machine?")

print("Response:")
print(response_coffee)
print()
print("Sources:")
print(sources_coffee)

Response:
According to the text, there are no specific safety instructions provided for using the machine in general. However, some guidelines are mentioned related to maintenance and cleaning:

* Do not pour keittintä (coffee maker) into any kind of liquid.
* Use Clean Drop -puhdistusliuosa (cleaning solution) only when instructed by its packaging.
* Follow the instructions on the packaging of Clean Drop -puhdistusliuokseen.

No explicit safety precautions or guidelines are mentioned in the text for using the machine.

Sources:
['moccamaster_manual_fi.pdf:10:3', 'moccamaster_manual_fi.pdf:9:0', 'moccamaster_manual_fi.pdf:2:4', 'moccamaster_manual_fi.pdf:1:4', 'moccamaster_manual_fi.pdf:9:1']


In [33]:
response_coffee_2, sources_coffee_2 = rag.query("How can I prepare coffee step by step?")

print("Response:")
print(response_coffee_2)
print()
print("Sources:")
print(sources_coffee_2)

Response:
The text doesn't provide a step-by-step guide to preparing coffee. However, it does list some key factors to consider when choosing a coffee maker:

1. Suodatuslämpötila (92–96 °C)
2. Veden ja kahvijauheen kontaktiaika eli saturaatioaika (4–6 minuuttia)
3. Puhdistuksen helppous
4. Pitkäikäisyys
5. Luotettavuus
6. Takuu

It also mentions that the coffee maker should be cleaned regularly, and provides some tips on how to do so:

1. Älä upota keitintä minkäänlaiseen nesteeseen.
2. Keitin on puhdistettava säännöllisin väliajoin.
3. Mikäli keitintä käytetään päivittäin, suosittelemme puhdistusta kahden viikon välein.
4. Käytä puhdistamiseen vain Clean Drop -puhdistusliuosta.
5. Noudata Clean Drop -puhdistusliuoksen pakkauksessa annettuja ohjeita.

However, it does not provide a detailed step-by-step guide on how to prepare coffee using the Moccamaster coffee maker.

Sources:
['moccamaster_manual_fi.pdf:2:5', 'moccamaster_manual_fi.pdf:1:5', 'moccamaster_manual_fi.pdf:2:4', 'moccam

## Question 3: UFO files

The file `ufo_files.zip` (available on Canvas) contains declassified CIA files pertaining to CIA's investigation into unidentified flying objects (UFOs).

Extract the package and add the documents into your `rag`. Then, query whatever you think would be interesting to ask about the documents, such as where and when sightings have occurred, have there been spikes, does the collection contain information about publicly famous cases (such as the Roswell Incident), about governmental programs relating to UFOs, patterns in sightings, international relations, and so on.

Look at the sources the model provides you. Do the sources support the inferences made by the model? Discuss, and again include at least one prompt and response pair. Write 2–4 paragraphs.

### Answer 3

We asked two questions about the incidents in Turkey. In the first query, we asked as "Have there been incidents in Turkey?". The model responded as yes according to the text. But when we checked the sources, we saw that they were not UFO sightings. There are other incidents in the texts. One of them was a criminal event in Turkey. The other one was a government criticism by a columnist. He was using "viewed from the outer space" and the model interpreted this as a UFO incident.

In the second query, we tried to make it more specific and asked "Have there been UFO sightings in Turkey?". This time model responded with no which was correct according to the texts. But it gave some examples from Central Asia and Caucasus. This shows that it managed to evaluate question geographically and told about the incidents in relatively closer areas.

It managed to check information from multiple sources in this question. It also uses multiple sources while generating the response. When we checked the sources, we saw that they were related to the information given in the response.

You can see our queries, responses to the queries and the sources that the model used to generate responses below.

In [34]:
import glob

rag.reset()

for filepath in glob.iglob('ufo_files/*.pdf'):
    rag.add_document(filepath)

In [35]:
response_ufo, sources_ufo = rag.query("Have there been incidents in Turkey?")

print("Response:")
print(response_ufo)
print()
print("Sources:")
print(sources_ufo)

Response:
Yes, according to the provided text, there have been incidents in Turkey. Specifically:

* Four Kurdish rebels and one police officer were killed in a clash in southeastern Turkey (date not specified).
* There was an unspecified number of UFO sightings reported, which was commented on by a columnist.

Additionally, the article mentions that the country "appears to have a government when viewed from outer space", implying that there may be issues with governance or legitimacy within the country.

Sources:
['ufo_files/C05516812.pdf:0:0', 'ufo_files/C05516766.pdf:30:2', 'ufo_files/C05516699.pdf:5:0', 'ufo_files/C05517843.pdf:4:1', 'ufo_files/C05517843.pdf:4:0']


In [37]:
response_ufo_2, sources_ufo_2 = rag.query("Have there been UFO sightings in Turkey?")

print("Response:")
print(response_ufo_2)
print()
print("Sources:")
print(sources_ufo_2)

Response:
No, there is no mention of UFO sightings in Turkey in the provided context. The text mentions that a report over video was made on art and international news, but it does not specifically mention Turkey or any UFO sightings in the country. It also mentions reports from Jamundi Valley (presumably in India) and other locations like Nardaran and Suvalan, which are likely to be in Central Asia or the Caucasus region, but there is no mention of Turkey.

Sources:
['ufo_files/C05517677.pdf:3:4', 'ufo_files/C05517675.pdf:0:1', 'ufo_files/C05517760.pdf:0:2', 'ufo_files/C05517492.pdf:2:1', 'ufo_files/C05517849.pdf:2:1']


## Question 4: Your own data

Provide some documents of your own to the system. The data can be arbitrary, but preferably something that is widely not considered publicly available or googleable. It can be also something completely made up for the purpose, like a short story. **Suggestion:** If you come from another discipline than computer science or data science, try using some data within your own discipline. Perhaps a suitable research article? Anything will do.

Present meaningful questions that can be answered with the data, and discuss the results. Write 2–4 paragraphs and again include at least one prompt and response pair.

### Answer 4

We have added a document about Kubernetes and its evolution. The paper mentions about Borg, Omega and Kubernetes, and their challenges and the lessons learned by engineers throughout their development. We asked what was the document about as a query. It could successfully understand the paper and its content. It mentioned that it's a technical discussion about Kubernetes.

We also asked what are the pros and cons of Kubernetes compared to other methods Borg and Omega as a query. It successfully mentioned that what are Kubernetes' pros are, such as simplicity, consistency, flexibility and being user friendly. According to the attached pdf it also found that Kubernetes is complex and making it harder to add specific features and it compared Kubernetes with other methods as well.

You can see our queries, responses to the queries and the sources that the model used to generate responses below.

In [38]:
rag.reset()
rag.add_document("papertask.pdf")

In [39]:
response_paper, sources_paper = rag.query("What is the document about?")

print("Response:")
print(response_paper)
print()
print("Sources:")
print(sources_paper)

Response:
The document appears to be a technical discussion or outline related to Kubernetes, a container orchestration system. It covers various topics such as container management, object metadata, API design, and dependencies between services. The tone suggests a technical documentation or a planning document for a project related to Kubernetes.

Sources:
['papertask.pdf:7:1', 'papertask.pdf:11:0', 'papertask.pdf:20:0', 'papertask.pdf:12:2', 'papertask.pdf:14:2']


In [40]:
response_paper_2, sources_paper_2 = rag.query("What are the pros and cons of Kubernetes compared to other methods?")

print("Response:")
print(response_paper_2)
print()
print("Sources:")
print(sources_paper_2)

Response:
Based on the provided context, here are the pros and cons of Kubernetes compared to other methods:

Pros of Kubernetes:

1. Simplicity: Kubernetes has a simpler system architecture as it uses generic tools that work across all objects, resulting in a consistent user experience.
2. Consistency: Kubernetes achieves consistency through decoupling and common design patterns, making it easier for developers to deploy and manage complex distributed systems.
3. Flexibility: Kubernetes allows users to add their own APIs dynamically, alongside the core functionality, providing flexibility and adaptability.
4. Developer-friendly: Kubernetes was designed with a focus on the experience of developers writing applications that run in a cluster, making it easy to deploy and manage complex distributed systems.

Cons of Kubernetes:

1. Complexity: Compared to Borg, Omega's client-side library provides more consistency, but this comes at the cost of added complexity for Kubernetes.
2. Over-eng